# 13 · Diagnosing failures

Method-of-lines computations fail in recognisable ways, and each failure has a test that identifies it and a
remedy. This notebook stages eight of them deliberately - instability from a non-dissipative stencil with the
wrong integrator, a boundary closure that puts eigenvalues in the right half-plane, ringing of a non-L-stable
method on rough data, negative concentrations from an oscillatory scheme feeding a nonlinear rate, a domain
too small for the solution, aliasing in an under-resolved spectral run, a source added after the operator
pinned a boundary, and a Newton iteration that converged to a wrong answer - and shows the symptom, the
diagnosis and the fix for each.

**What you will learn**
- recognise instability, boundary-induced growth, ringing, undershoot, boundary leakage, aliasing, pinning errors and spurious roots from their symptoms
- use the spectrum, conservation integrals, invariants and boundary values as diagnostics
- apply the remedy that matches the diagnosis

**Before you start:** notebooks 02, 07 and 10  ·  **Time:** about 75 min

In [1]:
try:                      # on Google Colab (or anywhere engmolines is missing): install it from GitHub
    import engmolines
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmolines"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engmolines import grids, boundary as bc, operators as op, integrators as ti, stencils as st, stiffness, analysis, problems as pr, spectral
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## 1. Growth from the imaginary axis: centred advection + explicit Euler

In [2]:
p = pr.advection_periodic(scheme="centred")
s = ti.explicit_rk(p.rhs, p.y0, (0, 0.5), p.grid.dx, ti.EULER, n_out=5)
print("max |u| per output:", np.round([np.max(np.abs(y)) for y in s.y], 2), " <- every mode grows, the sawtooth fastest")
lam = stiffness.spectrum(p.rhs, p.y0)
print(f"diagnosis: the spectrum is purely imaginary (max |Re| = {np.max(np.abs(lam.real)):.0e}) and Euler's stability region has no interior on the imaginary axis -> use RK4/Dormand-Prince (CFL 2.8), or an upwind/limited stencil")

max |u| per output: [  1.     1.25   1.74   2.76  12.24 222.48]  <- every mode grows, the sawtooth fastest
diagnosis: the spectrum is purely imaginary (max |Re| = 3e-10) and Euler's stability region has no interior on the imaginary axis -> use RK4/Dormand-Prince (CFL 2.8), or an upwind/limited stencil


The symptom is a sawtooth that grows without bound; the test is the spectrum of the right-hand side (purely
imaginary) against the integrator's stability region (which must enclose a piece of the imaginary axis). RK4
does, Euler and Heun do not; or change the stencil to one with dissipation.

## 2. A boundary closure with eigenvalues in the right half-plane

In [3]:
g = grids.cell_centred(0, 1, 100)
def rhs_bad(t, u):                                      # centred advection with an extrapolated ghost at the *inflow*: the wrong closure for a > 0
    up = np.r_[2 * u[0] - u[1], u, u[-1]]               # left ghost by extrapolation, right ghost zero-gradient
    return -(up[2:] - up[:-2]) / (2 * g.dx)
lam = stiffness.spectrum(rhs_bad, np.zeros(100))
print(f"largest real part of the spectrum: {lam.real.max():+.2f}  <- a growing boundary mode")
s = ti.dopri45(rhs_bad, np.exp(-((g.x - 0.3) / 0.05) ** 2), (0, 2.0), 1e-6, 1e-9, n_out=1, max_steps=20000)
print(f"after t = 2 (the pulse has long left the domain): max |u| = {np.max(np.abs(s.y[-1])):.1e}")
good = lambda t, u: op.advection(u, g, 1.0, bc.Dirichlet(0.0), bc.Neumann(0.0), t, "centred")   # noqa: E731
lam_g = stiffness.spectrum(good, np.zeros(100))
print(f"with the inflow value imposed instead: largest real part {lam_g.real.max():+.2e}")

largest real part of the spectrum: +2.25  <- a growing boundary mode
after t = 2 (the pulse has long left the domain): max |u| = 2.0e+01
with the inflow value imposed instead: largest real part -9.87e-04


A boundary treatment inconsistent with the direction of information flow puts an eigenvalue in the right
half-plane: the solution looks fine while the pulse is inside and then grows from the boundary, exponentially
and at any step size. The diagnosis is again the spectrum - a positive real part is never a time-step problem -
and the fix is the correct inflow condition.

## 3. Ringing: rough data and a method that is not L-stable

In [4]:
g = grids.uniform(0, 1, 40)
f = lambda t, u: op.diffusion(u, g, 1.0, bc.Dirichlet(0.0), bc.Dirichlet(0.0), t)   # noqa: E731
u0 = np.where((g.x > 0.4) & (g.x < 0.6), 1.0, 0.0)
tz = ti.trapezoidal(f, u0, (0, 0.05), 0.01, band=(1, 1), n_out=1)
be = ti.bdf(f, u0, (0, 0.05), 0.01, 1, band=(1, 1), n_out=1)
print(f"trapezoidal: min value {tz.y[-1].min():+.3f} (negative temperatures from a non-negative start); backward Euler: min {be.y[-1].min():+.4f}")
print(f"diagnosis: h lambda_max = {0.01 * (-4 / g.dx**2):.0f}, |R| = {abs((1 - 32) / (1 + 32)):.2f} for the trapezoidal rule: the sawtooth is not damped -> L-stable method, or two backward-Euler steps first (Rannacher)")

trapezoidal: min value -0.009 (negative temperatures from a non-negative start); backward Euler: min -0.0000
diagnosis: h lambda_max = -64, |R| = 0.94 for the trapezoidal rule: the sawtooth is not damped -> L-stable method, or two backward-Euler steps first (Rannacher)


Negative values from a non-negative start, oscillating from step to step, are the signature of a non-L-stable
method on data with high-frequency content. The test is |R(hλ_max)| near 1; the remedy is BDF, Radau, or a
Rannacher start.

## 4. Negative concentrations feeding a nonlinear rate

In [5]:
g = grids.cell_centred(0, 1, 200)
front = np.where(g.x < 0.3, 1.0, 0.0)
for sch in ("centred", "van-leer"):
    f = lambda t, c, sch=sch: op.advection(c, g, 1.0, bc.Periodic(), bc.Periodic(), t, sch) - 50.0 * c * (c - 1) * (c + 0.1)   # a cubic rate with a root at -0.1
    s = ti.dopri45(f, front, (0, 0.3), 1e-6, 1e-9, n_out=1)
    print(f"{sch:<9}: min c = {s.y[-1].min():+.3f}, max c = {s.y[-1].max():.3f}")

centred  : min c = -0.270, max c = 1.090


van-leer : min c = -0.000, max c = 1.000


A centred scheme's undershoot takes the concentration below zero; a rate law evaluated there can become a
source (here the cubic flips sign below −0.1) and the "negative mass" grows. A monotone scheme (upwind or a
limiter) keeps the state in its physical range, which nonlinear rates require.

## 5. A domain too small for the solution

In [6]:
p = pr.porous_medium(n=200)
g_small = grids.cell_centred(-3, 3, 120)
f_small = lambda t, u: op.diffusion(np.maximum(u, 0) ** 2, g_small, 1.0, bc.Dirichlet(0.0), bc.Dirichlet(0.0), t)   # noqa: E731
s = ti.bdf(f_small, p.exact(g_small.x, 0.0), (0, 1.0), 2e-3, 2, band=(1, 1), n_out=1)
print(f"domain [-3, 3]: mass changed by {g_small.integrate(s.y[-1]) - g_small.integrate(p.exact(g_small.x, 0.0)):+.3f} (the exact support reaches |x| = {np.sqrt(12 * 1.1**(2/3)):.2f}); on [-5, 5] the drift is 1e-13")
print("diagnosis: track the integral of u (or the value at the boundary) over time; a conserved quantity that changes is leaving through the boundary")

domain [-3, 3]: mass changed by -0.276 (the exact support reaches |x| = 3.58); on [-5, 5] the drift is 1e-13
diagnosis: track the integral of u (or the value at the boundary) over time; a conserved quantity that changes is leaving through the boundary


When the solution reaches a boundary that was meant to be far away, the Dirichlet wall absorbs it and a
conserved integral drifts. Monitor conservation and the boundary values; enlarge the domain, or use a
transparent condition.

## 6. Aliasing in an under-resolved spectral run

In [7]:
for n in (64, 80, 256):
    p = pr.kdv_two_solitons(n=n)
    with np.errstate(all="ignore"):                       # the under-resolved run is allowed to blow up
        s = p.solve("dopri", rtol=1e-8, atol=1e-10, n_out=1, allow_failure=True)
    mom = p.grid.integrate(s.y[-1] ** 2) - p.params["momentum"]
    print(f"n = {n:3d}: reached t = {s.t[-1]:.1f} ({s.message}); momentum invariant changed by {mom:+.2e}; max |u| {np.max(np.abs(s.y[-1])):.2g} (solitons' peak 2.0)")

n =  64: reached t = 7.0 (maximum steps reached); momentum invariant changed by +2.03e+306; max |u| 1.3e+153 (solitons' peak 2.0)


n =  80: reached t = 8.0 (ok); momentum invariant changed by +3.90e-03; max |u| 2 (solitons' peak 2.0)


n = 256: reached t = 8.0 (ok); momentum invariant changed by +5.95e-14; max |u| 2 (solitons' peak 2.0)


With too few points the product u u_x aliases energy into the resolved modes: at n = 80 the momentum
invariant drifts by 4×10⁻³ and the taller soliton's peak is wrong, at n = 64 the aliased energy feeds back
until the run blows up, and at n = 256 the invariants are conserved to 10⁻¹⁴. The test is the invariants (or the tail of the spectrum,
which must decay to rounding); the fix is resolution or the 2/3 rule.

## 7. A source added after the operator pinned a boundary

In [8]:
g = grids.uniform(0, 1, 50)
def rhs_wrong(t, T):
    return op.advection(T, g, 1.0, bc.Dirichlet(1.0), bc.Neumann(0.0), t, "upwind") - 2.0 * T     # the sink also acts on the inlet node
def rhs_right(t, T):
    out = op.advection(T, g, 1.0, bc.Dirichlet(1.0), bc.Neumann(0.0), t, "upwind") - 2.0 * T
    out[0] = 0.0
    return out
for name, f in (("source after operator", rhs_wrong), ("inlet re-pinned", rhs_right)):
    s = ti.bdf(f, np.r_[1.0, np.zeros(50)], (0, 5.0), 0.01, 2, band=(1, 1), n_out=1)
    print(f"{name:<22}: inlet value at the end {s.y[-1][0]:.4f} (imposed 1), outlet {s.y[-1][-1]:.4f} (exact e^-2 = {np.exp(-2):.4f})")

source after operator : inlet value at the end 0.0000 (imposed 1), outlet 0.0000 (exact e^-2 = 0.1353)
inlet re-pinned       : inlet value at the end 1.0000 (imposed 1), outlet 0.1407 (exact e^-2 = 0.1353)


The operator sets the inlet node's rate to zero; a source added afterwards moves it anyway, and the whole
profile shifts (the exchanger of notebook 11 came out at 0.92 instead of 0.77 for this reason). The test is to
print the boundary values over time; the fix is to re-pin them after assembling the right-hand side - or to
pass the source into the operator.

## 8. Newton converged - to the wrong answer

In [9]:
p = pr.nonlinear_conductivity()
s = p.solve("bdf", 0.02, order=2)
print(f"BDF2 at h = 0.02: Newton converged at every step; final state min {s.y[-1].min():.3f} (the conductivity 1 + u is {1 + s.y[-1].min():.1e} there), steady-state error {np.max(np.abs(s.y[-1] - p.exact(p.x))):.2f}")
s2 = p.solve("bdf", 0.005, order=2)
print(f"BDF2 at h = 0.005: steady-state error {np.max(np.abs(s2.y[-1] - p.exact(p.x))):.1e}")

BDF2 at h = 0.02: Newton converged at every step; final state min -1.000 (the conductivity 1 + u is 1.1e-16 there), steady-state error 1.99


BDF2 at h = 0.005: steady-state error 1.4e-05


A nonlinear implicit equation can have several roots, and a large step from rough data can put Newton in the
basin of a non-physical one - here a state where the conductivity vanishes, which the discrete equations
accept as an equilibrium. Convergence proves nothing by itself; a physical check (positivity, bounds,
conservation) and a step-size refinement study do. The library's fixed-step integrators report Newton
failures, but a converged wrong root they cannot see.

## Inside the diagnosis: the amplification of the sawtooth

In [10]:
p = pr.advection_periodic(scheme="centred")
sig = analysis.symbol("first_central", np.pi / 2, p.grid.dx)                 # the mode theta = pi/2 of the centred stencil
for method in ("euler", "rk4"):
    print(f"{method}: |R(h sigma)| at h = dx/a (Courant 1) = {abs(stiffness.stability_function(method, -p.grid.dx * sig)):.4f}")

euler: |R(h sigma)| at h = dx/a (Courant 1) = 1.4142
rk4: |R(h sigma)| at h = dx/a (Courant 1) = 0.9939


## Exercises
1. Take the heat equation with Neumann ends and impose the Neumann condition by a *first-order* one-sided ghost (u_ghost = u_0 - dx g) instead of the centred one. Measure the global order of accuracy and the conservation error. Which diagnostic reveals the problem first?
2. Integrate Fisher's equation with an explicit step just above the diffusion limit. Describe the symptom: where does the instability start, and how fast does it grow per step (compare with |R(h lambda_max)|)?
3. **Implement it yourself:** write a function `diagnose(rhs, y, band)` that reports the spectrum's largest real part, the stiffness ratio, the RK4 and Euler step limits, and whether the spectrum touches the imaginary axis; run it on five problems of the library.